---
title: Price spikes
execute:
  enabled: true
---

`q.indicator.price_spikes` flags an absolute close-to-close move above a multiple of simple rolling average true range, then classifies its volume confirmation.

`price_spike_event` uses 0 for no spike, 1 for a spike with high relative volume, and 2 for a spike without high relative volume. The explicit high, low, close, and volume inputs must describe one instrument and have identical indexes.

In [1]:
import pandas as pd

import qrt as q

data = q.data.datasets.load("aapl")
end = data.index.max()
data = data.loc[end - pd.DateOffset(years=5) :].copy()
result = q.indicator.price_spikes(
    data["high"], data["low"], data["close"], data["volume"]
)
events = data[["close", "volume"]].join(result)
events.loc[events["price_spike_event"].ne(0)].tail(10)

,close,volume,price_spike_event,relative_volume
datetime,,,,
2025-08-06,212.407333,108483100,1,1.993069
2025-08-08,228.443710,113854000,1,1.884262
2025-09-10,226.150192,83440800,1,1.709741
2025-09-22,255.357559,105517400,1,1.791227
2025-10-10,244.578064,61999100,2,1.125452
2025-10-20,261.500183,90483000,1,1.961588
2026-01-20,246.242508,80267500,1,1.743424
2026-02-02,269.509308,73913400,2,1.373737
2026-02-12,261.489105,81077200,2,1.375667


In [2]:
import plotly.graph_objects as go

figure = go.Figure()
figure.add_trace(go.Scatter(x=events.index, y=events["close"], name="AAPL close", line={"color": "#2457a7"}))
for code, name, color in [(1, "High-volume spike", "#16856b"), (2, "Low-volume spike", "#d04a35")]:
    selected = events.loc[events["price_spike_event"].eq(code)]
    figure.add_trace(
        go.Scatter(
            x=selected.index,
            y=selected["close"],
            name=name,
            mode="markers",
            marker={"color": color, "size": 8},
        )
    )
figure.update_layout(title="AAPL close-to-close spikes", yaxis_title="Close", hovermode="x unified")
figure.show(renderer="notebook_connected")